<a href="https://colab.research.google.com/github/roughhawkbit/digi-inno-road-prod/blob/main/analysis/ProcessQuestionsSeparatelyThenAggregate.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Setup

In [ ]:
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

In [ ]:
import matplotlib
import os
import pandas
import sklearn
import sys
import transformers

In [ ]:
if IN_COLAB:
  dirpath = '/content/digi-inno-road-prod'
  if not os.path.isdir(dirpath):
    # TODO git pull
    !git clone https://github.com/roughhawkbit/digi-inno-road-prod.git
  sys.path.insert(0,dirpath)
else:
  module_path = os.path.abspath(os.path.join('..'))
  if not module_path in sys.path:
      sys.path.insert(0, module_path)

In [ ]:
from innoprod.digital_readiness_score import DRS_LEVELS
from innoprod.plotting_tools import rand_jitter
from innoprod.sheet_tools import get_sheet_dfs
from innoprod.wrangling.msyh_data_sharing import wrangle_roadmaps

In [ ]:
data = get_sheet_dfs()
roadmaps_df = wrangle_roadmaps(data['Roadmaps'])

In [ ]:
qual_cols = [
    'Summary review of Edge Digital diagnostic report & current state and key improvement areas',
    'What are the internal barriers to growth? How do you intend to finance future growth? Are there sufficient leadership and management skills in the business to achieve your growth? What opportunities do you have to expand into new markets?',
    'Details of any existing Digital Strategy', # Check: Rob added in on 21st May
    'Level of current Strategic Digital Skills/knowledge in the business',
    'Level of current Technical Digital Skills/knowledge in the business',
    'Whether the business is already investing/adopting/utilising Industry 4.0 Technologies, with examples',
    'Summary of the identified problems, including Gap Analysis'
]

In [ ]:
model_name = "facebook/bart-large-mnli"
# model_name = "google-bert/bert-base-uncased"
# model_name = "pborchert/BusinessBERT"
classifier = transformers.pipeline("zero-shot-classification", model=model_name)
hypothesis_template = ("This company's digital readiness level is best described as: {}")

# Sandpit

In [ ]:
single_firm_responses = [q + ". " + r for q,r in zip(qual_cols, roadmaps_df[qual_cols].iloc[156].to_list())]
# single_firm_responses

In [ ]:
results = classifier(
      single_firm_responses,
      candidate_labels=DRS_LEVELS,
      hypothesis_template=hypothesis_template
)
# results

# Null results

In [ ]:
null_results = classifier(
      qual_cols,
      candidate_labels=DRS_LEVELS,
      hypothesis_template=hypothesis_template
)
# null_results

In [ ]:
def get_drs_level_score(row, drs_level):
    drs_label = DRS_LEVELS[drs_level - 1]
    index = row["labels"].index(drs_label)
    return row["scores"][index]

def results_to_df(results, null_results=None):
  results_df = pandas.DataFrame(results)
  for i in range(1,len(DRS_LEVELS)+1):
    col = f"DRS:{i}"
    drs_scores = results_df.apply(lambda row: get_drs_level_score(row, i), axis=1)
    if null_results is not None:
      drs_scores = drs_scores - null_results[col]
    results_df[col] = drs_scores

  results_df['Top DRS'] = results_df.filter(regex=r"DRS\:\d+").idxmax(axis=1).apply(lambda x: int(x.split(":")[1]))
  results_df['Confidence'] = results_df.filter(regex=r"DRS\:\d+").apply(lambda row: row.nlargest(1).values[0] - row.nlargest(2).values[-1], axis=1)
  return results_df.drop(columns=['labels', 'scores'])

null_results_df = results_to_df(null_results)
# null_results_df

In [ ]:
results_df = results_to_df(results, null_results=null_results_df)
# results_df

In [ ]:
def get_question_from_sequence(sequence):
  for q in qual_cols:
    if sequence.startswith(q):
      return q
  return None

results_df['question'] = results_df['sequence'].apply(get_question_from_sequence)
results_df

def get_most_confident_prediction(results, null_df):
  results_df = results_to_df(results, null_df)
  singlemost_idx = results_df['Confidence'].idxmax()
  singlemost_drs = int(results_df.iloc[singlemost_idx]['Top DRS'])
  singlemost_question = get_question_from_sequence(results_df.iloc[singlemost_idx]['sequence'])
  singlemost_drs_confidence = results_df['Confidence'].max()
  conf_df = results_df[['Top DRS', 'Confidence']].groupby('Top DRS').sum()
  combined_prediction = conf_df.idxmax(axis=0).tolist()[0]
  combined_prediction_confidence = conf_df['Confidence'].sort_values(ascending=False).to_list()
  combined_prediction_confidence = combined_prediction_confidence[0] - combined_prediction_confidence[1]
  return {
      'singlemost_drs': singlemost_drs,
      'singlemost_question': singlemost_question,
      'singlemost_drs_confidence': singlemost_drs_confidence,
      'combined_prediction': combined_prediction,
      'combined_prediction_confidence': combined_prediction_confidence
  }

get_most_confident_prediction(results, null_results_df)

In [ ]:
# all_seq is a list of lists: one list for each firm, and an item for each question-response within each sub-list
all_seq = roadmaps_df.apply(lambda row: [q + ". " + r for q,r in zip(qual_cols, row[qual_cols].to_list())], axis=1).to_list()
all_seq = [seq for firm_seqs in all_seq for seq in firm_seqs]
len(all_seq)


In [ ]:
pipeline_preds = classifier(
      all_seq,
      candidate_labels=DRS_LEVELS,
      hypothesis_template=hypothesis_template
)

In [ ]:
def unflatten_list(l, sublist_len):
  return [l[i:i+sublist_len] for i in range(0, len(l), sublist_len)]
pipeline_preds = unflatten_list(pipeline_preds, len(qual_cols))
len(pipeline_preds)

In [ ]:
predictions = [get_most_confident_prediction(results, null_results_df) for results in pipeline_preds]
# predictions

In [ ]:
roadmaps_df = pandas.concat([roadmaps_df, pandas.DataFrame.from_dict(predictions)], axis=1)

In [ ]:
roadmaps_df['singlemost_question'].value_counts()

In [ ]:
roadmaps_df[['Current Digital Readiness Score (refer to PAS:1040)', 'singlemost_drs', 'combined_prediction']].dropna().corr()

In [ ]:
fig, ax = matplotlib.pyplot.subplots(figsize=(6,6))
x_col = 'Current Digital Readiness Score (refer to PAS:1040)'
y_col = 'singlemost_drs'
size_col = 'singlemost_drs_confidence'

X = roadmaps_df[[x_col, y_col, size_col]].dropna()[[x_col]].to_numpy().reshape(-1, 1)
Y = roadmaps_df[[x_col, y_col, size_col]].dropna()[[y_col]].to_numpy().reshape(-1, 1)
S = roadmaps_df[[x_col, y_col, size_col]].dropna()[[size_col]].to_numpy().reshape(-1, 1)

lr_model = sklearn.linear_model.LinearRegression()
lr_model.fit(X, Y)

intercept = float(lr_model.intercept_[0])
coef = float(lr_model.coef_[0][0])

ax.scatter(rand_jitter(X), rand_jitter(Y), s=S*50)
ax.plot([1,len(DRS_LEVELS)], [intercept+coef, intercept+len(DRS_LEVELS)*coef])
ax.set_xlabel('Expert-assigned DRS')
ax.set_ylabel('Singlemost predicted DRS')
ax.set_xlim(1,len(DRS_LEVELS))
ax.set_ylim(1,len(DRS_LEVELS))

In [ ]:
lr_model.score(X, Y)

In [ ]:
fig, ax = matplotlib.pyplot.subplots(figsize=(6,6))
x_col = 'Current Digital Readiness Score (refer to PAS:1040)'
y_col = 'combined_prediction'
size_col = 'combined_prediction_confidence'

X = roadmaps_df[[x_col, y_col, size_col]].dropna()[[x_col]].to_numpy().reshape(-1, 1)
Y = roadmaps_df[[x_col, y_col, size_col]].dropna()[[y_col]].to_numpy().reshape(-1, 1)
S = roadmaps_df[[x_col, y_col, size_col]].dropna()[[size_col]].to_numpy().reshape(-1, 1)

lr_model = sklearn.linear_model.LinearRegression()
lr_model.fit(X, Y)

intercept = float(lr_model.intercept_[0])
coef = float(lr_model.coef_[0][0])

ax.scatter(rand_jitter(X), rand_jitter(Y), s=S*50)
ax.plot([1,len(DRS_LEVELS)], [intercept+coef, intercept+len(DRS_LEVELS)*coef])
ax.set_xlabel('Expert-assigned DRS')
ax.set_ylabel('Combined prediction DRS')
ax.set_xlim(1,len(DRS_LEVELS))
ax.set_ylim(1,len(DRS_LEVELS))